# Сны грунта: сеть предлагает прочтение, грунт проверяет

Ученик (`rows.py` + `records.py`) находит смысл предложений новой задачи поиском: от минут до часов на задачу, и почти всё это время уходит на прочтения, далёкие от верного. Если прочтения предложений даны, та же задача выучивается за десятую долю секунды. Здесь их угадывает сеть, выращенная на «снах» грунта — до всяких слов:

- **сны** (`relive.py`): задачи, которые ученик уже выучил целиком, проживаются заново — в своих рассказах и в новых, которых не было, — и в вариациях: одно-два предложения прочитаны иначе, ответы пересчитаны грунтом. Автора у таких задач нет, их сколько угодно;
- **сеть** (`guess.py`): трансформер читает рассказы новой задачи (слов не видит: только какие виды предложений есть, какие вещи в них стоят и какими были ответы) и для каждого вида предложений называет прочтения, которые стоит пробовать первыми;
- **грунт** проверяет предложенное точно, как любую догадку, и чинит: по очереди пробует первые варианты для каждого вида предложений. Где ничего не подошло — обычный поиск;
- **та же сеть без грунта**: вторым выходом она сама отвечает на вопросы нового рассказа, имея перед собой только предыдущие рассказы задачи с ответами. Это то, что сеть, выращенная на снах грунта, может сама; первое — что она может, когда исполняет и проверяет грунт.

Что измеряется: наборы H, I, J и K (48 задач других авторов) в обучении сети не участвуют. На каждой задаче — её первые 20 рассказов: объясняет ли первое же предложение сети все вопросы, объясняет ли после починки, и что это прочтение даёт на следующих сорока рассказах; рядом — память слово в слово (`memory.py`) и ответы сети без грунта на тех же сорока рассказах.

Как запускать: среда выполнения T4 (или лучше), «Выполнить всё». Обучение сети — `MX_MINUTES` минут (по умолчанию 90); оборванный запуск при повторном выполнении продолжается с сохранённого места. Короткая проверка, что всё запускается: `%env MX_QUICK=1` до первой ячейки. Нужен вывод ячеек с итогами (строки, начинающиеся с «tasks» и «all the tasks», и таблица в конце).

In [ ]:
import os, subprocess, sys, json, re, time, glob
ROOT = os.environ.get('MX_ROOT', '/content')
QUICK = bool(os.environ.get('MX_QUICK'))
MINUTES = float(os.environ.get('MX_MINUTES', 0.5 if QUICK else 90))
WORK = ROOT + '/mx-dreams'
os.makedirs(WORK, exist_ok=True)
if os.path.exists(ROOT + '/MX-Project'):
    subprocess.run(['git', '-C', ROOT + '/MX-Project', 'pull', '-q'])
else:
    subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/kireikosasha/MX-Project', ROOT + '/MX-Project'], check=True)
M5 = ROOT + '/MX-Project/research_m5'
os.environ['PYTHONHASHSEED'] = '0'
!python -u {M5}/colab/mind.py --work {WORK} --fetch | tail -1
for name in list('ABCDEFGHIJKL') + ['M2']:
    !python {M5}/newtasks.py --set {name} --out {WORK}/new/{name} | tail -1

## Грунт без смысла предложений: записи и вспоминание (`recall.py`)

Эта часть видеокарты не требует. Рассказ хранится как записи — вид предложения и вещи в нём; смысла у предложений нет. Каждый вид вопроса сам находит пробой и точной проверкой, на какие виды предложений смотреть и чем каждый из них ему служит. Способов смотреть три: значение (последнее держится), связь (одно перед другим), количество; у предложения может быть ключ (о чём оно: север, цвет) и время (число в нём), а вопрос может быть о другом времени, чем «сейчас». Учится по 20 рассказам и может спросить ответы ещё 20; дальше — проверка на отложенных рассказах. Строка на задачу: сколько верно на проверке, в скольких видах вопросов уверен и сколько верно на них, время.

In [ ]:
import concurrent.futures
sets = {'en': f'{WORK}/tasks_1-20_v1-2/en', **{x: f'{WORK}/new/{x}' for x in 'ABCDEFGHIJK'}}
jobs = [('en', '1,2,3'), ('K', '1,4'), ('J', '9')] if QUICK else [(name, '') for name in sets]
def one(job):
    name, tasks = job
    out = subprocess.run(f"cd {M5} && {sys.executable} recall.py --data {sets[name]}" + (f" --tasks {tasks}" if tasks else ""), shell=True, capture_output=True, text=True).stdout
    open(f'{WORK}/recall_{name}.txt', 'w').write(out)
    return name, out
got, secs = {}, 0.0
with concurrent.futures.ThreadPoolExecutor(os.cpu_count() or 2) as pool:
    for name, out in pool.map(one, jobs):
        for line in out.splitlines():
            m = re.match(r'task +(\d+): .*\(([\d.]+) s\)', line)
            if m:
                secs += float(m.group(2))
            m = re.match(r'test +(\d+): (\d+)/(\d+) = ([\d.]+)%; sure of (\d+) of (\d+) kinds of questions, (\d+)/(\d+)', line)
            if m:
                got[(name, int(m.group(1)))] = (float(m.group(4)), int(m.group(7)), int(m.group(8)), int(m.group(3)))
for name in sets:
    ks = sorted(k for n, k in got if n == name)
    if ks:
        print(f"{name:2s}: до 95 % и выше {sum(got[(name, k)][0] >= 95 for k in ks)} из {len(ks)} | " + ' '.join(f"{k}:{got[(name, k)][0]:.0f}" for k in ks))
v = [x[0] for x in got.values()]
print(f"всего задач {len(v)}: до 95 % и выше {sum(x >= 95 for x in v)}, среднее {sum(v) / len(v):.1f} %; уверен в {100 * sum(x[2] for x in got.values()) / sum(x[3] for x in got.values()):.0f} % вопросов, "
      f"на них верно {100 * sum(x[1] for x in got.values()) / max(1, sum(x[2] for x in got.values())):.1f} %; время учения {secs:.0f} с")

## Память как слой, обучаемый градиентом, и читатель слов (`soft.py`)

Видеокарты тоже не требует. Те же памяти сделаны слоем сети: пока учится, каждый вид предложений играет все свои роли сразу (мягко), градиент ответов идёт сквозь память назад к выбору ролей; затем выбор затвердевает и задача проверяется точно. Первая часть ячейки — одна задача с нуля, чистым градиентом. Вторая — читатель слов (`words.pt`, 49 тысяч весов, выучен сквозь слой по одним ответам на 73 задачах): новую задачу он читает по словам её предложений; печатается, сколько вопросов объяснено сразу, после 100 шагов градиента и после точной доводки. Набор M2 он не видел (слепая проверка этапа 31). С `%env MX_FULL=1` читатель слов учится заново: слой проходит все наборы (на двух ядрах это часы), затем 4000 шагов обучения.

In [ ]:
sets['L'], sets['M2'] = f'{WORK}/new/L', f'{WORK}/new/M2'
print('--- одна задача с нуля, градиентом:')
first = '1' if QUICK else '1,2,6,9,15'
en, m2 = sets['en'], sets['M2']
!cd {M5} && python soft.py --data {en} --tasks {first} | cut -c1-200
if os.environ.get('MX_FULL') and not QUICK:
    os.makedirs(f'{WORK}/soft', exist_ok=True)
    for name in sets:
        folder = sets[name]
        !cd {M5} && python soft.py --data {folder} --show | grep -v "^    beginning" > {WORK}/soft/{name}.txt
    pairs = ','.join(k + '=' + v for k, v in sets.items())
    !cd {M5} && python soft.py --results {WORK}/soft --sets {pairs} --hold M2 --teach 4001 --words {WORK}/words.pt | grep -E "^[0-9]+ tasks|step [0-9]*000,"
words = f'{WORK}/words.pt' if os.path.exists(f'{WORK}/words.pt') else f'{M5}/words.pt'
print('--- набор M2 по словам:')
only = '--tasks 9,11' if QUICK else ''
!cd {M5} && python soft.py --words {words} --data {m2} {only} | grep -E "read from its words|^test|more things" | cut -c1-200

## Петля, которая не плывёт (`loop.py`)

Память узлов: у узла вид, значение и три поля-указателя. Команда — «шаг по полю» или «идти по полю до узла такого вида и сказать: куда пришёл / сколько шагов / сумму значений по пути / наибольшее». Смысл 51 команды выучивается только по ответам. Ядро: состояние петли — распределение над парами (узел, число в регистре), оборот — чтение памяти; учится только маршрутизация (714 чисел), потом выбор затвердевает. Учат на ответах не дальше 4 шагов, проверяют на 8–40 шагах и на 1024 оборотах. Соперник — зацикленный трансформер (вход на каждом обороте, норма после оборота, случайное число оборотов); на процессоре за два часа он не выучивает задачу и на учебной глубине, поэтому честное сравнение — здесь, на видеокарте: `MX_MINUTES` минут обучения. Последняя строка таблицы — тот, кто всегда называет самый частый ответ столбца: для «наибольшего» на длинном пути это почти всегда 9, а «сколько шагов» в столбце одно и то же — эти две строки у трансформера надо читать с этой поправкой.

In [ ]:
!cd {M5} && python loop.py --minutes {0.3 if QUICK else MINUTES} --width {96 if QUICK else 256} --layers {2 if QUICK else 4} | grep -v "^looped, step\|^machine, step"

## Чужие мерила: StepGame и CLUTRR (`walk.py`)

Видеокарты не требует. StepGame (Shi, Zhang, Lipani, 2022): рассказ о том, как предметы стоят друг относительно друга, до десяти предложений на ответ; лучшая сеть статьи даёт 86 % при k = 1 и 21 % при k = 10. CLUTRR (Sinha и др., 2019): цепочки родства, учат на 2–3 звеньях, проверяют до 10; графовая сеть с вниманием падает с 91 % до 45 %. Здесь рассказ — связи между вещами, ответ — прогулка по связям; как связи соединяются, выучивается одним из двух способов (регистр из целых чисел, мягкий выбор и градиент сквозь прогулку — или таблица пар, выученная подсчётом, и разбор пути), а какой годится набору, решает отложенная пятая часть рассказов. Ожидается: StepGame 95–98 % при всех k (все оставшиеся ошибки — предложения, которые сам набор использует для двух отношений; вне их 100 %), CLUTRR около 99 %. Последний запуск: четыре пятых видов предложений StepGame с ответами не показываются; их смысл узнаётся из рассказов без ответов — регистр обязан сойтись по двум путям между одними и теми же вещами (ожидается: почти все прочитаны, рассказы с ними решаются на 95–99 %, по словам — на 31–44 %). И самый жёсткий вариант (этап 40): рассказы пересказаны шаблонами двенадцати авторов; учебные — словами одних авторов, проверочные — других, ни одно предложение проверки не встречалось. Ожидается: по словам около 70 %, после чтения рассказов без ответов — 100 % при каждом k; на третьей группе авторов, не читанной вовсе, чтение по словам вырастает с 58 % до 71 %. Последний запуск — исходный тест StepGame, шаблоны которого написаны людьми и ни разу не показаны с ответом: ожидается около 69 % по словам и около 96 % после чтения рассказов без ответов.

In [ ]:
if not os.path.exists(f'{WORK}/StepGame'):
    !git clone -q --depth 1 --filter=blob:none --sparse https://github.com/ZhengxiangShi/StepGame {WORK}/StepGame && git -C {WORK}/StepGame sparse-checkout set Dataset/TrainVersion
few = '--stories 3000 --begin 2' if QUICK else ''
!cd {M5} && python walk.py --data {WORK}/StepGame/Dataset/TrainVersion {few} | grep -v "^begun"
for part in ('train', 'test'):
    if not os.path.exists(f'{WORK}/clutrr_{part}.csv'):
        !wget -q -O {WORK}/clutrr_{part}.csv https://raw.githubusercontent.com/kliang5/CLUTRR_huggingface_dataset/main/gen_train23_test2to10/{part}.csv
!cd {M5} && python walk.py --clutrr {WORK}/clutrr {'--begin 2' if QUICK else ''}
print('--- четыре пятых видов предложений StepGame не показаны с ответами; читаются из рассказов без ответов:')
!cd {M5} && python walk.py --data {WORK}/StepGame/Dataset/TrainVersion --hold 0.8 {'--begin 2' if QUICK else ''} | grep -v "^    "
print('--- рассказы словами других авторов: учебные - авторы 1-4, без ответов - авторы 5-8, третья группа (9-12) не читается вовсе:')
!cd {M5} && python anew.py --data {WORK}/StepGame/Dataset/TrainVersion --out {WORK}/anew_ab --teach 1,2,3,4 --test 5,6,7,8 --seed 3 | tail -2
!cd {M5} && python anew.py --data {WORK}/StepGame/Dataset/TrainVersion --out {WORK}/anew_ac --teach 1,2,3,4 --test 9,10,11,12 --seed 3 | tail -1
!cd {M5} && python walk.py --data {WORK}/anew_ab --foreign --then {WORK}/anew_ac --seed 3 {'--begin 2' if QUICK else ''}
print('--- исходный тест StepGame (шаблоны написаны людьми) без единого ответа на его языке: учебные рассказы - словами 24 авторов:')
every = ','.join(str(n) for n in range(1, 25))
!cd {M5} && python anew.py --data {WORK}/StepGame/Dataset/TrainVersion --out {WORK}/anew_all --teach {every} --test 1 --seed 5 | tail -1
!cd {M5} && python walk.py --data {WORK}/anew_all --foreign --tests {WORK}/StepGame/Dataset/TrainVersion --seed 5 {'--begin 2' if QUICK else ''}

## Соперник на видеокарте: трансформер на StepGame (`rival.py`)

Тот же протокол, что у `walk.py` выше: учиться на 50 000 чистых рассказов с k от 1 до 5, проверяться на рассказах с помехами при k от 1 до 10. Два вида: обычный трансформер (6 слоёв, ширина 256, около 5 млн весов — в две с половиной тысячи раз больше, чем выучивает `walk.py`) и зацикленный (один блок, прокручиваемый по кругу, вход на каждом обороте, норма после оборота; проверяется с тем же числом оборотов, вдвое и вчетверо большим). Каждый учится `MX_MINUTES / 2` минут. В статье набора лучшая сеть давала 86 % при k = 1 и 21 % при k = 10, Universal Transformer — 45 % и 12 %; `walk.py` — 98 % и 95 %. Что здесь проверяется: держит ли трансформер, обученный как следует, длину, которой не видел.

In [ ]:
small = '--width 64 --layers 2 --heads 4 --tried 100' if QUICK else ''
each = 0.3 if QUICK else MINUTES / 2
!cd {M5} && python rival.py --data {WORK}/StepGame/Dataset/TrainVersion --minutes {each} {small} | grep -v "^step"
!cd {M5} && python rival.py --data {WORK}/StepGame/Dataset/TrainVersion --minutes {each} --looped --turns 8 {small} | grep -v "^step"

## Задачи, как они были поняты

Из готовых результатов ученика (`results/rows/*.txt`) берутся прочтения предложений; программы вопросов грунт находит заново одной подгонкой — секунды на все задачи.

In [ ]:
R = M5 + '/results/rows'
specs = [f'{WORK}/tasks_1-20_v1-2/en:{R}/rerun_en.txt'] + [f'{WORK}/new/{x}:{R}/rerun_{x}.txt' for x in 'ABCDEFG']
specs += [f'{WORK}/new/H:{R}/bridge_H_sure.txt', f'{WORK}/new/I:{R}/bridge_I.txt', f'{WORK}/new/J:{R}/bridge_J.txt', f'{WORK}/new/K:{R}/bridge_K.txt']
!cd {M5} && python relive.py --tasks {','.join(specs)} --out {WORK}/lived.pkl | tail -1

## Сеть учится на снах и пробуется на чужих задачах

Наборы H, I, J и K отложены: сны строятся только из задач bAbI и наборов A–G. В конце — по строке на каждую отложенную задачу и два итога: по задачам, которые ученик выучил поиском целиком, и по остальным.

In [ ]:
size = '--width 64 --layers 2 --heads 4 --most 640 --stories 8 --least 5 --batch 8 --look 300 --every 20 --few 4' if QUICK else '--width 384 --layers 8 --heads 8 --batch 24 --every 200'
!cd {M5} && python -u guess.py --lived {WORK}/lived.pkl --hold H,I,J,K --minutes {MINUTES} {size} --out {WORK}/guess.pt 2>&1 | tee -a {WORK}/guess.txt | grep -v "^step [0-9]*[1-9]00,"

## Необязательно: петля вместо высокой укладки

Та же сеть, но слоёв два, и они проходятся четыре раза подряд с теми же весами: вычислений столько же, сколько у восьми слоёв, весов вчетверо меньше. Отвечает на вопрос, что важнее для рассуждения над состоянием — число разных слоёв или число шагов. Включается отдельно: `%env MX_LOOPS=1` до первой ячейки (ещё столько же минут обучения).

In [ ]:
if os.environ.get('MX_LOOPS') or QUICK:
    loop = '--width 64 --layers 1 --loops 2 --heads 4 --most 640 --stories 8 --least 5 --batch 8 --look 300 --every 20 --few 4' if QUICK else '--width 384 --layers 2 --loops 4 --heads 8 --batch 24 --every 200'
    !cd {M5} && python -u guess.py --lived {WORK}/lived.pkl --hold H,I,J,K --minutes {MINUTES} {loop} --out {WORK}/guess_loop.pt 2>&1 | tee -a {WORK}/guess_loop.txt | grep -v "^step [0-9]*[1-9]00,"
else:
    print('пропущено: чтобы сравнить с петлёй, выполните %env MX_LOOPS=1 до первой ячейки')

## Сеть отвечает сама: грунт через топологию

То же, но без грунта вообще (`mind.py`): сеть читает рассказы новой задачи слева направо и на каждом вопросе говорит ответ; слов по-прежнему не видит, имена видов предложений и вещей раздаются заново в каждой задаче, так что смысл можно взять только из предыдущих рассказов. Сравниваются сборки одного размера и времени обучения:

- **обычная** — слои один над другим;
- **с памятью записей** (`--names 2`) — каждое утверждение рассказа остаётся записью (вид предложения и его вещи), и в любом месте сеть может вспомнить последние записи, где названа вещь. Встроено только это: имя находится точно (метки слов не обучаются), последнее упоминание — первым, ответ можно взять словом из вспомненной записи. Какое имя спросить, у какого вида предложений, что взять из записи и когда пойти по цепочке — выучивается;
- **с памятью записей, на снах самого грунта** (`--names 2 --ground`) — та же сборка, но учится не на задачах, которые ученик выучил, а на задачах, которых никто не писал: грунт записей (`recall.py`) сам бросает жребий — виды вещей, виды предложений со случайными ролями в трёх видах памяти (значение, связь, количество), виды вопросов — и сам на них отвечает. Ни одной задачи, написанной человеком, сеть до проверки не видит: это прямая проверка мысли «вырастить на алгоритмических инвариантах до текста»;
- необязательно **петля** (`%env MX_LOOPS=1`) — два слоя, пройденные четыре раза, с той же памятью записей.

Проверка — те же отложенные наборы H, I, J и K: перед сетью первые рассказы задачи с ответами, ответить надо на вопросы следующих; затем каждой задаче даётся ещё 300 малых шагов на её собственных двадцати рассказах, и проверка повторяется. Каждая сборка учится `MX_MINUTES` минут.

In [ ]:
size = '--width 64 --layers 2 --heads 4 --most 640 --stories 8 --least 5 --batch 8 --every 20 --few 3 --tune 3' if QUICK else '--width 384 --layers 8 --heads 8 --batch 24 --every 200 --tune 300'
builds = [('с памятью записей', 'mind_names', '--names 2'), ('обычная', 'mind_plain', ''), ('с памятью записей, на снах самого грунта', 'mind_ground', '--names 2 --ground')]
if os.environ.get('MX_LOOPS') and not QUICK:
    builds.append(('петля с памятью записей', 'mind_loop', '--names 2 --loops 4 --layers 2'))
for label, name, more in builds:
    print('=====', label)
    !cd {M5} && python -u mind.py --lived {WORK}/lived.pkl --hold H,I,J,K --minutes {MINUTES} {size} {more} --out {WORK}/{name}.pt 2>&1 | tee -a {WORK}/{name}.txt | grep -E "weights|^tasks|^all the tasks|^step [0-9]*000,"

## Ученик с сетью и без неё

Те же отложенные задачи целиком, как в слепых прогонах: 20 рассказов и до 20 запрошенных, проверка на тесте. С сетью (`--guess --fast`): если предложенное прочтение объясняет все вопросы первых рассказов и запрошенных, оно берётся без поиска; иначе поиск идёт как обычно, начиная с предложенного. Без сети — числа слепых прогонов из `results/rows/bridge_*.txt`. Поиск считает на процессоре и на двух ядрах Colab занимает несколько часов, поэтому часть включается отдельно: `%env MX_FULL=1` до первой ячейки.

In [ ]:
def table(path):
    out = {}
    for l in open(path):
        m = re.match(r'task +(\d+):.*\((\d+) s, ', l)
        if m:
            k, secs, quick = int(m.group(1)), int(m.group(2)), 'taken at once' in l
        m = re.match(r'deep +(\d+): (\d+)/(\d+)', l)
        if m:
            out[int(m.group(1))] = (100 * int(m.group(2)) / int(m.group(3)), secs, quick)
    return out

if os.environ.get('MX_FULL') or QUICK:
    !cd {M5} && python memory.py --out {WORK}/memory_AG.json results/rows/rerun_*.txt | tail -1
    sets = {'H': 'bridge_H_sure.txt'} if QUICK else {'H': 'bridge_H_sure.txt', 'I': 'bridge_I.txt', 'J': 'bridge_J.txt'}
    halves = ('1',) if QUICK else ('1,3,5,7,9,11', '2,4,6,8,10,12')
    for name, old in sets.items():
        jobs = [subprocess.Popen(f"cd {M5} && {sys.executable} -u rows.py --data {WORK}/new/{name} --ask 20 --limit {40000 if QUICK else 300000} --tasks {h} --show "
                                 f"--memory {WORK}/memory_AG.json --guess {WORK}/guess.pt --fast > {WORK}/with_{name}_{i}.txt 2>&1", shell=True) for i, h in enumerate(halves)]
        for j in jobs:
            j.wait()
        new, was = {}, table(f'{R}/{old}')
        for i in range(len(halves)):
            new.update(table(f'{WORK}/with_{name}_{i}.txt'))
        print(f'набор {name}: задача: верно % (время, с) с сетью | без сети')
        for k in sorted(new):
            w = was.get(k)
            print(f"  {name}{k}: {new[k][0]:.1f} ({new[k][1]} с{', взято сразу' if new[k][2] else ''}) | " + (f"{w[0]:.1f} ({w[1]} с)" if w else 'не досчиталась'))
        ks = sorted(k for k in new if k in was)
        print(f"  по задачам, досчитавшимся в обоих прогонах: среднее {sum(new[k][0] for k in ks) / len(ks):.1f} | {sum(was[k][0] for k in ks) / len(ks):.1f}; "
              f"до 95 % и выше {sum(new[k][0] >= 95 for k in ks)} | {sum(was[k][0] >= 95 for k in ks)} из {len(ks)}; "
              f"взято сразу {sum(new[k][2] for k in ks)}; время {sum(new[k][1] for k in ks)} | {sum(was[k][1] for k in ks)} с")
else:
    print('пропущено: чтобы посчитать ученика целиком, выполните %env MX_FULL=1 до первой ячейки')